In [8]:
import GtoTmodel
import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers =10 # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text

In [10]:
# Check if GPU is available and set the device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [11]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim,
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [12]:
from Circuits import Circuits
circuits = Circuits()


Loading dataset files...
Loaded dataset files successfully.


In [13]:
# Define the file path to load the model and hyperparameters
load_path = "model_checkpoint.pth"

# Load the checkpoint
checkpoint = torch.load(load_path)

# Restore the model state and hyperparameters
model.load_state_dict(checkpoint['model_state_dict'])

# Restore hyperparameters if needed
embed_dim = checkpoint['embed_dim']
num_heads = checkpoint['num_heads']
num_layers = checkpoint['num_layers']
dropout = checkpoint['dropout']
text_vocab_size = checkpoint['text_vocab_size']
graph_input_dim = checkpoint['graph_input_dim']

print(f"Model and optimizer state loaded from {load_path}")

C:\Users\MSI\AppData\Local\Temp\ipykernel_14724\3870851804.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(load_path)


Model and optimizer state loaded from model_checkpoint.pth


In [66]:
model.eval()
index=torch.randint(0, len(circuits.graphs), (1,)).item() # Random index to test
graph=torch.tensor(circuits.graphs[index],dtype=torch.float32).to(device)
text=torch.tensor(circuits.component_indices[index]).to(device)
graph.shape,text.shape
graph =F.pad(graph, (0, 310 - graph.size(0) if graph.size(1) < 310 else 0, 0, 0), mode='constant', value=9)
graph.size()
test_graph = graph.view(1,graph.size(0),310).to(torch.float32)
test_text = text.view(1,text.size(0)) 
circuits.get_component_fromlist(test_text[0].tolist())

['VDD',
 'VSS',
 'VIN1',
 'VOUT1',
 'VB1',
 'VB2',
 'VB3',
 'VCLK1',
 'VCLK2',
 'NM1',
 'NM1_D',
 'NM1_G',
 'NM1_S',
 'NM1_B',
 'NM2',
 'NM2_D',
 'NM2_G',
 'NM2_S',
 'NM2_B',
 'NM3',
 'NM3_D',
 'NM3_G',
 'NM3_S',
 'NM3_B',
 'NM4',
 'NM4_D',
 'NM4_G',
 'NM4_S',
 'NM4_B',
 'NM5',
 'NM5_D',
 'NM5_G',
 'NM5_S',
 'NM5_B',
 'NM6',
 'NM6_D',
 'NM6_G',
 'NM6_S',
 'NM6_B',
 'NM7',
 'NM7_D',
 'NM7_G',
 'NM7_S',
 'NM7_B',
 'PM1',
 'PM1_D',
 'PM1_G',
 'PM1_S',
 'PM1_B',
 'PM2',
 'PM2_D',
 'PM2_G',
 'PM2_S',
 'PM2_B',
 'PM3',
 'PM3_D',
 'PM3_G',
 'PM3_S',
 'PM3_B',
 'PM4',
 'PM4_D',
 'PM4_G',
 'PM4_S',
 'PM4_B',
 'NM8',
 'NM8_D',
 'NM8_G',
 'NM8_S',
 'NM8_B',
 'NM9',
 'NM9_D',
 'NM9_G',
 'NM9_S',
 'NM9_B',
 'PM5',
 'PM5_D',
 'PM5_G',
 'PM5_S',
 'PM5_B',
 'NM10',
 'NM10_D',
 'NM10_G',
 'NM10_S',
 'NM10_B',
 'NM11',
 'NM11_D',
 'NM11_G',
 'NM11_S',
 'NM11_B',
 'PM6',
 'PM6_D',
 'PM6_G',
 'PM6_S',
 'PM6_B',
 'PM7',
 'PM7_D',
 'PM7_G',
 'PM7_S',
 'PM7_B',
 'NM12',
 'NM12_D',
 'NM12_G',
 'NM12_S',
 'NM1

In [67]:


with torch.no_grad(): 
        if test_text.dim() == 1:
            test_text = test_text.unsqueeze(0)

        if test_graph.dim() == 1:
            test_graph = test_graph.unsqueeze(0)

        # Forward pass through the model
        output = model(test_graph, test_text[:, :-1])  # Exclude the last token for input
        # print("Output Shape:", output.shape)

        # Select the prediction for the last timestep
        last_timestep_output = output[:, -1, :]
        predicted_token = torch.argmax(last_timestep_output, dim=-1)
        # Get the top five predictions for the last timestep
        top_five_predictions = torch.topk(last_timestep_output, 5, dim=-1).indices.squeeze(0)

        # Convert the top five predictions to actual tokens
        top_five_tokens = [circuits.get_component(token.item()) for token in top_five_predictions]

        print("Top Five Predicted Tokens:", top_five_tokens)

print("Predicted Token:", circuits.get_component(predicted_token.item()))  # Convert to actual token
print("Actual Next Token:", circuits.get_component(int(test_text[:, -1])))  # Compare with the actual next token

Top Five Predicted Tokens: ['NM27_D', 'TRANSMISSION_GATE4_A', 'L9', 'PM6_B', 'NM21_S']
Predicted Token: NM27_D
Actual Next Token: C2_N
